In [1]:
import json
import datetime
from dateutil import parser
import os

In [2]:
with open("results_robust_gpt-4o-mini.json", "r") as file:
    example = json.load(file)

In [3]:
example['individual_results']

[{'sample_id': 0,
  'question': 'When did Caroline go to the LGBTQ support group?',
  'prediction': '8 May, 2023',
  'reference': '7 May 2023',
  'category': 2,
  'metrics': {'exact_match': 0,
   'f1': 0.6666666666666666,
   'rouge1_f': 0.6666666666666666,
   'rouge2_f': 0.5,
   'rougeL_f': 0.6666666666666666,
   'bleu1': 0.5,
   'bleu2': 0.12909944487358058,
   'bleu3': 0.09635409769034631,
   'bleu4': 0.09554427922043669,
   'bert_precision': 0.9760338068008423,
   'bert_recall': 0.9805994033813477,
   'bert_f1': 0.97831130027771,
   'meteor': 0.16666666666666666,
   'sbert_similarity': 0.8955168128013611}},
 {'sample_id': 0,
  'question': 'When did Melanie paint a sunrise?',
  'prediction': 'Last year',
  'reference': 2022,
  'category': 2,
  'metrics': {'exact_match': 0,
   'f1': 0.0,
   'rouge1_f': 0.0,
   'rouge2_f': 0.0,
   'rougeL_f': 0.0,
   'bleu1': 0,
   'bleu2': 0,
   'bleu3': 0,
   'bleu4': 0,
   'bert_precision': 0.882870078086853,
   'bert_recall': 0.8763823509216309,
  

In [4]:
example['aggregate_metrics'].keys()

dict_keys(['overall', 'category_1', 'category_2', 'category_3', 'category_4', 'category_5'])

In [5]:
metrics = list(example['aggregate_metrics']['overall'].keys())

In [6]:
remove = [
    'rouge1_f',
    'bleu2',
    'bleu3',
    'bleu4',
    'bert_precision',
    'bert_recall'
]

for metric in remove:
    metrics.remove(metric)

In [7]:
metrics[5:7] = metrics[5:7][::-1]

In [8]:
METRICS = {}

for metric in metrics:
    if 'rouge' in metric:
        metric_name = metric.split('_')[0]
    else:
        metric_name = metric
    if "rouge" in metric or "bleu" in metric:
        metric_name = '-'.join([metric_name[:-1].upper(), metric_name[-1].upper()])
    elif '_' in metric_name:
        metric_name = metric.split('_')
        sep = '-' if 'f1' in metric else " "
        if 'bert' in metric:
            metric_name = sep.join([metric_name[0].upper(), metric_name[-1].capitalize()])
        else:
            metric_name = sep.join([metric_name[0].capitalize(), metric_name[-1].capitalize()])
    else:
        metric_name = metric_name.upper()
    METRICS[metric] = metric_name

METRICS

{'exact_match': 'Exact Match',
 'f1': 'F1',
 'rouge2_f': 'ROUGE-2',
 'rougeL_f': 'ROUGE-L',
 'bleu1': 'BLEU-1',
 'meteor': 'METEOR',
 'bert_f1': 'BERT-F1',
 'sbert_similarity': 'SBERT Similarity'}

In [9]:
categories = [f'category_{i+1}' for i in range(5)] + ['overall']
category_names = ["Multi Hop", "Temporal", "Open Domain", "Single Hop", "Adversial", "Average"]
CATEGORIES = {
    category: name
    for name, category in zip(category_names, categories)
}

In [10]:
original = {
    "category_1": {  # Multi Hop
        "gpt-4o-mini": {
            "exact_match": "--",
            "f1": 27.02,
            "bleu1": 20.09,
            "bert_f1": "--",
            "rouge2_f": 10.61,
            "rougeL_f": 25.86,
            "meteor": 16.36,
            "sbert_similarity": 49.46,
        },
        "Qwen2.5-1.5B-Instruct": {
            "exact_match": "--",
            "f1": 18.23,
            "bleu1": 11.94,
            "bert_f1": "--",
            "rouge2_f": 4.88,
            "rougeL_f": 17.94,
            "meteor": 9.49,
            "sbert_similarity": 43.49,
        },
        "Qwen2.5-3B-Instruct": {
            "exact_match": "--",
            "f1": 12.57,
            "bleu1": 9.01,
            "bert_f1": "--",
            "rouge2_f": 2.91,
            "rougeL_f": 12.42,
            "meteor": 6.25,
            "sbert_similarity": 33.72,
        },
        "Llama-3.2-1B-Instruct": {
            "exact_match": "--",
            "f1": 19.06,
            "bleu1": 11.71,
            "bert_f1": "--",
            "rouge2_f": 4.82,
            "rougeL_f": 19.31,
            "meteor": 9.01,
            "sbert_similarity": 45.16,
        },
        "Llama-3.2-3B-Instruct": {
            "exact_match": "--",
            "f1": 17.44,
            "bleu1": 11.74,
            "bert_f1": "--",
            "rouge2_f": 6.02,
            "rougeL_f": 17.62,
            "meteor": 9.74,
            "sbert_similarity": 39.32,
        }
    },
    "category_2": {  # Temporal
        "gpt-4o-mini": {
            "exact_match": "--",
            "f1": 45.85,
            "bleu1": 36.67,
            "bert_f1": "--",
            "rouge2_f": 21.39,
            "rougeL_f": 44.27,
            "meteor": 23.43,
            "sbert_similarity": 70.49,
        },
        "Qwen2.5-1.5B-Instruct": {
            "exact_match": "--",
            "f1": 24.32,
            "bleu1": 19.74,
            "bert_f1": "--",
            "rouge2_f": 5.88,
            "rougeL_f": 27.23,
            "meteor": 11.92,
            "sbert_similarity": 61.65,
        },
        "Qwen2.5-3B-Instruct": {
            "exact_match": "--",
            "f1": 27.59,
            "bleu1": 25.07,
            "bert_f1": "--",
            "rouge2_f": 8.11,
            "rougeL_f": 27.74,
            "meteor": 14.04,
            "sbert_similarity": 62.54,
        },
        "Llama-3.2-1B-Instruct": {
            "exact_match": "--",
            "f1": 17.80,
            "bleu1": 10.28,
            "bert_f1": "--",
            "rouge2_f": 1.84,
            "rougeL_f": 20.47,
            "meteor": 7.50,
            "sbert_similarity": 54.79,
        },
        "Llama-3.2-3B-Instruct": {
            "exact_match": "--",
            "f1": 26.38,
            "bleu1": 19.50,
            "bert_f1": "--",
            "rouge2_f": 7.93,
            "rougeL_f": 27.97,
            "meteor": 13.19,
            "sbert_similarity": 59.70,
        }
    },
    "category_3": {  # Open Domain
        "gpt-4o-mini": {
            "exact_match": "--",
            "f1": 45.85,
            "bleu1": 36.67,
            "bert_f1": "--",
            "rouge2_f": 3.42,
            "rougeL_f": 12.09,
            "meteor": 8.36,
            "sbert_similarity": 38.48,
        },
        "Qwen2.5-1.5B-Instruct": {
            "exact_match": "--",
            "f1": 24.32,
            "bleu1": 19.74,
            "bert_f1": "--",
            "rouge2_f": 3.44,
            "rougeL_f": 16.87,
            "meteor": 9.11,
            "sbert_similarity": 42.58,
        },
        "Qwen2.5-3B-Instruct": {
            "exact_match": "--",
            "f1": 27.59,
            "bleu1": 25.07,
            "bert_f1": "--",
            "rouge2_f": 1.51,
            "rougeL_f": 7.51,
            "meteor": 6.56,
            "sbert_similarity": 30.60,
        },
        "Llama-3.2-1B-Instruct": {
            "exact_match": "--",
            "f1": 17.80,
            "bleu1": 10.28,
            "bert_f1": "--",
            "rouge2_f": 5.99,
            "rougeL_f": 18.49,
            "meteor": 8.30,
            "sbert_similarity": 43.42,
        },
        "Llama-3.2-3B-Instruct": {
            "exact_match": "--",
            "f1": 26.38,
            "bleu1": 19.50,
            "bert_f1": "--",
            "rouge2_f": 5.38,
            "rougeL_f": 13.00,
            "meteor": 8.09,
            "sbert_similarity": 32.27,
        }
    },
    "category_4": {  # Single Hop
        "gpt-4o-mini": {
            "exact_match": "--",
            "f1": 45.85,
            "bleu1": 36.67,
            "bert_f1": "--",
            "rouge2_f": 29.50,
            "rougeL_f": 45.18,
            "meteor": 42.32,
            "sbert_similarity": 59.38,
        },
        "Qwen2.5-1.5B-Instruct": {
            "exact_match": "--",
            "f1": 24.32,
            "bleu1": 19.74,
            "bert_f1": "--",
            "rouge2_f": 12.32,
            "rougeL_f": 24.38,
            "meteor": 19.69,
            "sbert_similarity": 41.93,
        },
        "Qwen2.5-3B-Instruct": {
            "exact_match": "--",
            "f1": 27.59,
            "bleu1": 25.07,
            "bert_f1": "--",
            "rouge2_f": 8.80,
            "rougeL_f": 17.57,
            "meteor": 15.98,
            "sbert_similarity": 33.98,
        },
        "Llama-3.2-1B-Instruct": {
            "exact_match": "--",
            "f1": 17.80,
            "bleu1": 10.28,
            "bert_f1": "--",
            "rouge2_f": 14.82,
            "rougeL_f": 29.78,
            "meteor": 22.46,
            "sbert_similarity": 47.07,
        },
        "Llama-3.2-3B-Instruct": {
            "exact_match": "--",
            "f1": 26.38,
            "bleu1": 19.50,
            "bert_f1": "--",
            "rouge2_f": 16.89,
            "rougeL_f": 28.55,
            "meteor": 24.30,
            "sbert_similarity": 42.86,
        }
    },
    "category_5": {  # Adversarial
        "gpt-4o-mini": {
            "exact_match": "--",
            "f1": 45.85,
            "bleu1": 36.67,
            "bert_f1": "--",
            "rouge2_f": 42.62,
            "rougeL_f": 50.04,
            "meteor": 45.64,
            "sbert_similarity": 53.26,
        },
        "Qwen2.5-1.5B-Instruct": {
            "exact_match": "--",
            "f1": 24.32,
            "bleu1": 19.74,
            "bert_f1": "--",
            "rouge2_f": 36.32,
            "rougeL_f": 46.60,
            "meteor": 40.64,
            "sbert_similarity": 52.44,
        },
        "Qwen2.5-3B-Instruct": {
            "exact_match": "--",
            "f1": 27.59,
            "bleu1": 25.07,
            "bert_f1": "--",
            "rouge2_f": 21.39,
            "rougeL_f": 27.98,
            "meteor": 27.36,
            "sbert_similarity": 33.72,
        },
        "Llama-3.2-1B-Instruct": {
            "exact_match": "--",
            "f1": 17.80,
            "bleu1": 10.28,
            "bert_f1": "--",
            "rouge2_f": 46.76,
            "rougeL_f": 60.23,
            "meteor": 53.72,
            "sbert_similarity": 68.00,
        },
        "Llama-3.2-3B-Instruct": {
            "exact_match": "--",
            "f1": 26.38,
            "bleu1": 19.50,
            "bert_f1": "--",
            "rouge2_f": 35.48,
            "rougeL_f": 42.25,
            "meteor": 39.74,
            "sbert_similarity": 46.76,
        }
    },
}

In [24]:
def format_best(rep_score, base_score):
    if rep_score * 100 > base_score:
        rep_score = f"\\textbf{{{rep_score * 100:.2f}}}"
    else:
        base_score = f"\\textbf{{{base_score:.2f}}}"
    return rep_score, base_score

In [94]:
def format_table(tabular):
    table = f"""\\begin{{table}}[]
    \\centering
    \\resizebox{{\\linewidth}}{{!}}{{
        \\begin{{tabular}}{{l|cc|cccc|cc}}
        {"".join(tabular)}
        \\end{{tabular}}
}}
\\end{{table}}
    """.strip()
    
    print(table)

In [107]:
def write_table(results, category):
    tabular = ["\\toprule"]
    header = "\\textbf{Model}\t"
    for metric in METRICS.values():
        header += f"\t&\t\\textbf{{{metric}}}"
    header += "\t\\\\"
    category_name = CATEGORIES[category]
    tabular.append(f"\\multicolumn{{9}}{{c}}{{\\textbf{{{category_name}}}}}\t\\\\")
    tabular.append('\\midrule')
    tabular.append(header)
    models = set()
    if category == 'overall':
        tabular.append('\\midrule')
    for result in results:
        model = result['model'].split('/')[-1]
        if original.get(category) is not None:
            baseline = original[category][model]
            if not model.split('-')[0] in models:
                tabular.append('\\midrule')
                models.add(model.split('-')[0])
        else:
            baseline = None
        if "gpt" in model:
            model = model.replace(model[:3], model[:3].upper())
        row_base = f"{model}\t"
        row_rep = "\\quad{+ Reproduction}\t" if baseline is not None else f"{model}\t"
        scores = result['aggregate_metrics'][category]
        for metric in METRICS.keys():
            score = scores[metric]['mean']
            if baseline is not None:
                base_score = baseline[metric]
                if isinstance(base_score, float):
                    score, base_score = format_best(score, base_score)
                row_base += f"&\t{base_score}\t"
            if isinstance(score, str):
                row_rep += f"&\t{score}\t"
            else:
                row_rep += f"&\t{score * 100:.2f}\t"
        row_base += '\\\\'
        row_rep += '\\\\'
        if baseline is not None:
            tabular.append(row_base)
        tabular.append(row_rep)
    tabular.append('\\bottomrule')

    return (
        '\t' + row + '\n' if not i == 0 else row + '\n' for i, row in enumerate(tabular)
    )

In [108]:
results_files = [file for file in os.listdir() if file.startswith("results_robust")]
results = {}
for result_file in results_files:
    with open(result_file, "r") as file:
        results[result_file] = json.load(file)

In [109]:
for category in categories:
    format_table(write_table(list(results.values()), category))
    print('\n\n')

\begin{table}[]
    \centering
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|cc|cccc|cc}
        \toprule
	\multicolumn{9}{c}{\textbf{Multi Hop}}	\\
	\midrule
	\textbf{Model}		&	\textbf{Exact Match}	&	\textbf{F1}	&	\textbf{ROUGE-2}	&	\textbf{ROUGE-L}	&	\textbf{BLEU-1}	&	\textbf{METEOR}	&	\textbf{BERT-F1}	&	\textbf{SBERT Similarity}	\\
	\midrule
	GPT-4o-mini	&	--	&	\textbf{27.02}	&	\textbf{10.61}	&	\textbf{25.86}	&	\textbf{20.09}	&	\textbf{16.36}	&	--	&	\textbf{49.46}	\\
	\quad{+ Reproduction}	&	0.71	&	23.17	&	8.83	&	22.30	&	16.52	&	12.87	&	88.08	&	43.37	\\
	\midrule
	Qwen2.5-3B-Instruct	&	--	&	12.57	&	2.91	&	12.42	&	9.01	&	6.25	&	--	&	33.72	\\
	\quad{+ Reproduction}	&	4.26	&	\textbf{20.25}	&	\textbf{6.16}	&	\textbf{19.96}	&	\textbf{12.03}	&	\textbf{8.95}	&	88.31	&	\textbf{45.72}	\\
	Qwen2.5-1.5B-Instruct	&	--	&	\textbf{18.23}	&	4.88	&	\textbf{17.94}	&	11.94	&	\textbf{9.49}	&	--	&	\textbf{43.49}	\\
	\quad{+ Reproduction}	&	1.06	&	18.11	&	\textbf{5.38}	&	17.92	&	\textbf{13.36}

In [16]:
keys = ["model", "dataset", "total_questions", "individual_results"]

In [17]:
result_dir = "individual_results"
os.makedirs(result_dir, exist_ok=True)
for result_file, result in results.items():
    ind_result = {
        key: result[key]
        for key in keys
    }
    with open(f"{result_dir}/{result_file}", "w") as file:
        json.dump(ind_result, file, indent=2)